## SQL Data Setup

This notebook loads the processed compensation datasets into a SQLite database for business-focused SQL analysis.

In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

occupation_scorecard = pd.read_csv(
    "../data/processed/occupation_compensation_scorecard.csv",
    dtype={"noc_code": str}
)

occupation_trends = pd.read_csv(
    "../data/processed/occupation_market_trends.csv",
    dtype={"noc_code": str}
)

industry_scorecard = pd.read_csv(
    "../data/processed/industry_compensation_scorecard.csv"
)

industry_trends = pd.read_csv(
    "../data/processed/industry_market_trends.csv"
)

In [2]:
occupation_scorecard.head()

,noc_code,Occupation,nova_scotia_wage,canada_wage,market_gap_pct,market_position,ns_growth_pct,canada_growth_pct,growth_difference
0,111,Professional occupations in finance,36.20,43.00,-15.81,Below Market,35.68,33.58,2.10
1,112,Professional occupations in business,40.64,41.41,-1.86,Near Market,50.97,26.67,24.30
2,211,Professional occupations in natural sciences,40.85,44.87,-8.96,Below Market,22.56,24.99,-2.43
3,212,Professional occupations in applied sciences (...,46.04,50.48,-8.80,Below Market,36.82,38.64,-1.82
4,213,Professional occupations in engineering,48.06,52.00,-7.58,Below Market,26.31,26.18,0.13


In [3]:
occupation_scorecard.head()

,noc_code,Occupation,nova_scotia_wage,canada_wage,market_gap_pct,market_position,ns_growth_pct,canada_growth_pct,growth_difference
0,111,Professional occupations in finance,36.20,43.00,-15.81,Below Market,35.68,33.58,2.10
1,112,Professional occupations in business,40.64,41.41,-1.86,Near Market,50.97,26.67,24.30
2,211,Professional occupations in natural sciences,40.85,44.87,-8.96,Below Market,22.56,24.99,-2.43
3,212,Professional occupations in applied sciences (...,46.04,50.48,-8.80,Below Market,36.82,38.64,-1.82
4,213,Professional occupations in engineering,48.06,52.00,-7.58,Below Market,26.31,26.18,0.13


In [4]:
#Create the SQLite database
db_path = "../sql/compensation_analysis.db"

conn = sqlite3.connect(db_path)

In [5]:
#write the DataFrames into SQL table
occupation_scorecard.to_sql(
    "occupation_scorecard",
    conn,
    if_exists="replace",
    index=False
)

occupation_trends.to_sql(
    "occupation_trends",
    conn,
    if_exists="replace",
    index=False
)

industry_scorecard.to_sql(
    "industry_scorecard",
    conn,
    if_exists="replace",
    index=False
)

industry_trends.to_sql(
    "industry_trends",
    conn,
    if_exists="replace",
    index=False
)

print("SQLite database created successfully.")

SQLite database created successfully.


In [6]:
conn.close()

In [7]:
conn = sqlite3.connect("../sql/compensation_analysis.db")

In [8]:
pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table';
""", conn)

,name
0,occupation_scorecard
1,occupation_trends
2,industry_scorecard
3,industry_trends


#### Question 1

Which occupations have the largest negative market gaps in Nova Scotia in 2025?

In [9]:
query = """
SELECT
    noc_code,
    Occupation,
    nova_scotia_wage,
    canada_wage,
    market_gap_pct
FROM occupation_scorecard
ORDER BY market_gap_pct ASC;
"""

result = pd.read_sql(query, conn)

result

,noc_code,Occupation,nova_scotia_wage,canada_wage,market_gap_pct
0,111,Professional occupations in finance,36.20,43.00,-15.81
1,413,Professional occupations in social and communi...,36.06,40.00,-9.85
2,412,Professional occupations in education services,43.27,47.84,-9.55
3,211,Professional occupations in natural sciences,40.85,44.87,-8.96
4,212,Professional occupations in applied sciences (...,46.04,50.48,-8.80
5,213,Professional occupations in engineering,48.06,52.00,-7.58
6,312,Therapy and assessment professionals,46.63,49.00,-4.84
7,414,Professional occupations in government services,43.94,46.15,-4.79
8,421,Occupations in front-line public protection se...,48.08,50.42,-4.64
9,112,Professional occupations in business,40.64,41.41,-1.86


### Business Interpretation

The query ranks occupational groups according to the difference between Nova Scotia and Canadian median hourly wages.

**Key Findings**

- Professional occupations in finance had the largest negative market gap at **15.81% below** the Canadian benchmark.
- Professional occupations in social and community services, education, natural sciences, and engineering also showed notable wage gaps.
- Health-related occupations and legal professions were the only occupational groups with wages exceeding the Canadian benchmark.
- The results indicate that compensation competitiveness varies considerably across occupational groups rather than uniformly across the labour market.

#### Question 2
Which occupations are currently at or above the Canadian market benchmark?

In [10]:
query = """
SELECT
    noc_code,
    Occupation,
    nova_scotia_wage,
    canada_wage,
    market_gap_pct,
    market_position
FROM occupation_scorecard
WHERE market_position IN ('Near Market', 'Above Market')
ORDER BY market_gap_pct DESC;
"""

result = pd.read_sql(query, conn)

result

,noc_code,Occupation,nova_scotia_wage,canada_wage,market_gap_pct,market_position
0,411,Professional occupations in law,74.52,62.50,19.23,Above Market
1,313,Nursing and allied health professionals,48.00,46.00,4.35,Near Market
2,311,Health treating and consultation services prof...,52.50,51.00,2.94,Near Market
3,422,"Paraprofessional occupations in legal, social,...",27.00,26.92,0.30,Near Market
4,112,Professional occupations in business,40.64,41.41,-1.86,Near Market
5,421,Occupations in front-line public protection se...,48.08,50.42,-4.64,Near Market
6,414,Professional occupations in government services,43.94,46.15,-4.79,Near Market
7,312,Therapy and assessment professionals,46.63,49.00,-4.84,Near Market


### Business Interpretation

This query identifies occupational groups whose median hourly wages are close to or exceed the Canadian benchmark.

**Key Findings**

- **Professional occupations in law** were the only occupational group classified as **Above Market**, with Nova Scotia wages **19.23% above** the national benchmark.
- Seven occupational groups were classified as **Near Market**, indicating that Nova Scotia compensation is generally within ±5% of the Canadian benchmark.
- Health-related occupations, including **Nursing and Allied Health Professionals** and **Health Treating and Consultation Services Professionals**, demonstrated strong market competitiveness.
- Business, government services, public protection, and therapy occupations also remained relatively close to the national benchmark.

These occupational groups may require less immediate compensation review than occupations with larger negative market gaps.

#### Question 3

How can occupational groups be classified into compensation risk categories?

In [11]:
query = """
SELECT
    noc_code,
    Occupation,
    market_gap_pct,

    CASE
        WHEN market_gap_pct < -10 THEN 'High Risk'
        WHEN market_gap_pct < -5 THEN 'Moderate Risk'
        WHEN market_gap_pct < 5 THEN 'Competitive'
        ELSE 'Above Market'
    END AS compensation_risk

FROM occupation_scorecard

ORDER BY market_gap_pct;
"""

result = pd.read_sql(query, conn)

result

,noc_code,Occupation,market_gap_pct,compensation_risk
0,111,Professional occupations in finance,-15.81,High Risk
1,413,Professional occupations in social and communi...,-9.85,Moderate Risk
2,412,Professional occupations in education services,-9.55,Moderate Risk
3,211,Professional occupations in natural sciences,-8.96,Moderate Risk
4,212,Professional occupations in applied sciences (...,-8.80,Moderate Risk
5,213,Professional occupations in engineering,-7.58,Moderate Risk
6,312,Therapy and assessment professionals,-4.84,Competitive
7,414,Professional occupations in government services,-4.79,Competitive
8,421,Occupations in front-line public protection se...,-4.64,Competitive
9,112,Professional occupations in business,-1.86,Competitive


### Business Interpretation

This query classifies occupational groups according to compensation competitiveness using predefined market-gap thresholds.

**Key Findings**

- **Professional occupations in finance** were classified as **High Risk**, with a market gap of **15.81% below** the Canadian benchmark.
- Five occupational groups were classified as **Moderate Risk**, including education, engineering, natural sciences, applied sciences, and social and community services.
- Most occupational groups were classified as **Competitive**, indicating relatively small differences from the Canadian benchmark.
- **Professional occupations in law** were the only occupational group classified as **Above Market**, with Nova Scotia wages exceeding the national benchmark.

This type of classification can help compensation analysts prioritize occupational groups for further market review.

#### Question 4

What is the average market gap for occupations by market position?

In [12]:
query = """
SELECT

    market_position,

    COUNT(*) AS number_of_occupations,

    ROUND(
        AVG(market_gap_pct),
        2
    ) AS average_market_gap

FROM occupation_scorecard

GROUP BY market_position;
"""

result = pd.read_sql(query, conn)

result

,market_position,number_of_occupations,average_market_gap
0,Above Market,1,19.23
1,Below Market,6,-10.09
2,Near Market,7,-1.22


### Business Interpretation

This query summarizes occupational market competitiveness by calculating the number of occupational groups and the average market gap within each compensation category.

**Key Findings**

- **6 occupational groups** were classified as **Below Market**, with an average compensation gap of **10.09% below** the Canadian benchmark.
- **7 occupational groups** were classified as **Near Market**, with an average gap of only **1.22% below** the national benchmark.
- Only **1 occupational group** was classified as **Above Market**, with Nova Scotia wages averaging **19.23% above** the Canadian benchmark.

Overall, half of the occupational groups remain below the national benchmark, while the majority of the remaining groups are relatively competitive. These results suggest that compensation review efforts should prioritize occupations within the **Below Market** category.

#### Question 5

Which occupational groups have remained below the Canadian benchmark on average over the last five years (2021–2025)?

In [13]:
query = """
SELECT

    noc_code,

    Occupation,

    ROUND(
        AVG(market_gap_pct),
        2
    ) AS average_market_gap

FROM occupation_trends

WHERE year BETWEEN 2021 AND 2025

GROUP BY
    noc_code,
    Occupation

HAVING AVG(market_gap_pct) < 0

ORDER BY average_market_gap ASC;
"""

result = pd.read_sql(query, conn)

result

,noc_code,Occupation,average_market_gap
0,111,Professional occupations in finance,-15.93
1,212,Professional occupations in applied sciences (...,-10.32
2,413,Professional occupations in social and communi...,-9.76
3,412,Professional occupations in education services,-9.46
4,112,Professional occupations in business,-9.32
5,213,Professional occupations in engineering,-8.79
6,422,"Paraprofessional occupations in legal, social,...",-7.79
7,312,Therapy and assessment professionals,-7.78
8,421,Occupations in front-line public protection se...,-5.76
9,414,Professional occupations in government services,-4.52


### Business Interpretation

This query identifies occupational groups that remained below the Canadian compensation benchmark on average between **2021 and 2025**.

**Key Findings**

- **Professional occupations in finance** had the largest persistent compensation gap, averaging **15.93% below** the Canadian benchmark over the five-year period.
- **Professional occupations in applied sciences**, **social and community services**, **education services**, and **business** also experienced sustained negative market gaps averaging between **9% and 10% below** the national benchmark.
- **Engineering**, **therapy and assessment**, **public protection**, and **government services** remained consistently below the Canadian benchmark, although with smaller average gaps.
- Health-related occupations showed comparatively stronger market competitiveness, with **Health Treating and Consultation Services Professionals** averaging only **2.03% below** and **Nursing and Allied Health Professionals** averaging **0.62% below** the national benchmark.

These results suggest that several occupational groups have experienced persistent compensation gaps rather than short-term fluctuations, making them higher priorities for labour-market monitoring and compensation review.

#### Question

What are the five highest-paying occupational groups in Nova Scotia (2025)?

In [14]:
query = """
SELECT
    noc_code,
    Occupation,
    nova_scotia_wage
FROM occupation_scorecard
ORDER BY nova_scotia_wage DESC
LIMIT 5;
"""

result = pd.read_sql(query, conn)

result

,noc_code,Occupation,nova_scotia_wage
0,411,Professional occupations in law,74.52
1,311,Health treating and consultation services prof...,52.50
2,421,Occupations in front-line public protection se...,48.08
3,213,Professional occupations in engineering,48.06
4,313,Nursing and allied health professionals,48.00


### Business Interpretation

This query identifies the five highest-paying occupational groups in Nova Scotia based on 2025 median hourly wages.

**Key Findings**

- **Professional occupations in law** recorded the highest median hourly wage among the occupational groups analyzed.
- Health-related occupations, including **Health Treating and Consultation Services Professionals** and **Nursing and Allied Health Professionals**, were also among the highest-paying occupations.
- Engineering and public protection occupations remained within the top five, reflecting comparatively strong compensation levels within Nova Scotia.

Identifying the highest-paying occupational groups helps establish internal wage benchmarks and provides useful context for labour-market comparisons.

#### Question 7
Which occupations are currently below market and also had slower wage growth than Canada?

In [15]:
query = """
SELECT
    s.noc_code,
    s.Occupation,
    s.market_gap_pct,
    ROUND(
        (
            MAX(CASE WHEN t.year = 2025 THEN t.nova_scotia_wage END) -
            MAX(CASE WHEN t.year = 2015 THEN t.nova_scotia_wage END)
        )
        /
        MAX(CASE WHEN t.year = 2015 THEN t.nova_scotia_wage END)
        * 100,
        2
    ) AS ns_growth_pct,

    ROUND(
        (
            MAX(CASE WHEN t.year = 2025 THEN t.canada_wage END) -
            MAX(CASE WHEN t.year = 2015 THEN t.canada_wage END)
        )
        /
        MAX(CASE WHEN t.year = 2015 THEN t.canada_wage END)
        * 100,
        2
    ) AS canada_growth_pct

FROM occupation_scorecard AS s

JOIN occupation_trends AS t
    ON s.noc_code = t.noc_code

WHERE s.market_gap_pct < 0

GROUP BY
    s.noc_code,
    s.Occupation,
    s.market_gap_pct

HAVING ns_growth_pct < canada_growth_pct

ORDER BY s.market_gap_pct ASC;
"""

result = pd.read_sql(query, conn)

result

,noc_code,Occupation,market_gap_pct,ns_growth_pct,canada_growth_pct
0,413,Professional occupations in social and communi...,-9.85,15.39,33.33
1,412,Professional occupations in education services,-9.55,27.00,29.23
2,211,Professional occupations in natural sciences,-8.96,22.56,24.99
3,212,Professional occupations in applied sciences (...,-8.80,36.82,38.64
4,421,Occupations in front-line public protection se...,-4.64,23.28,23.37


### Business Interpretation

This query identifies occupational groups that satisfy two important conditions:

1. They are currently below the Canadian compensation benchmark.
2. Their wage growth between 2015 and 2025 has been slower than the national average.

**Key Findings**

- Professional occupations in **social and community services** demonstrated the greatest concern, with both a substantial current market gap and significantly slower wage growth than Canada.
- **Education**, **natural sciences**, and **applied sciences** also experienced slower wage growth while remaining below the national benchmark.
- **Front-line public protection occupations** remained slightly below market and experienced wage growth nearly equal to the national average.

These occupations may warrant additional compensation review because slower wage growth has not been sufficient to close existing market gaps.

#### Question 8
Which occupations should be prioritized for compensation review based on both current market gap and recent five-year average gap?

In [16]:
query = """
WITH five_year_gap AS (

    SELECT
        noc_code,
        Occupation,
        ROUND(AVG(market_gap_pct), 2) AS avg_gap_2021_2025

    FROM occupation_trends

    WHERE year BETWEEN 2021 AND 2025

    GROUP BY
        noc_code,
        Occupation
)

SELECT
    s.noc_code,
    s.Occupation,
    s.market_gap_pct AS current_gap_2025,
    f.avg_gap_2021_2025,

    CASE
        WHEN s.market_gap_pct < -10
             AND f.avg_gap_2021_2025 < -10
            THEN 'Priority Review'

        WHEN s.market_gap_pct < -5
             OR f.avg_gap_2021_2025 < -5
            THEN 'Monitor'

        ELSE 'Competitive'
    END AS review_status

FROM occupation_scorecard AS s

JOIN five_year_gap AS f
    ON s.noc_code = f.noc_code

ORDER BY current_gap_2025 ASC;
"""

result = pd.read_sql(query, conn)

result

,noc_code,Occupation,current_gap_2025,avg_gap_2021_2025,review_status
0,111,Professional occupations in finance,-15.81,-15.93,Priority Review
1,413,Professional occupations in social and communi...,-9.85,-9.76,Monitor
2,412,Professional occupations in education services,-9.55,-9.46,Monitor
3,211,Professional occupations in natural sciences,-8.96,-4.06,Monitor
4,212,Professional occupations in applied sciences (...,-8.80,-10.32,Monitor
5,213,Professional occupations in engineering,-7.58,-8.79,Monitor
6,312,Therapy and assessment professionals,-4.84,-7.78,Monitor
7,414,Professional occupations in government services,-4.79,-4.52,Competitive
8,421,Occupations in front-line public protection se...,-4.64,-5.76,Monitor
9,112,Professional occupations in business,-1.86,-9.32,Monitor


### Business Interpretation

This query combines current market competitiveness with historical market performance to identify occupational groups that may require compensation review.

**Key Findings**

- **Professional occupations in finance** were identified as the only **Priority Review** group, reflecting both a large current market gap (**15.81% below** Canada) and a persistent five-year average gap (**15.93% below** Canada).
- Several occupational groups, including engineering, education, applied sciences, social and community services, and therapy professions, were classified as **Monitor**, indicating continued labour-market competitiveness concerns.
- Business and paraprofessional occupations illustrate how historical trends can differ from current conditions. Although their 2025 market gaps are relatively small, their five-year averages suggest longer-term competitive challenges.
- Government services, health professions, and legal occupations were classified as **Competitive**, reflecting stronger overall market positioning.

This analysis demonstrates how combining current and historical compensation indicators provides a more robust basis for compensation planning than relying on a single-year comparison.

#### Question9 
How do Nova Scotia occupational groups rank by median hourly wage in 2025?

In [17]:
query = """
SELECT
    noc_code,
    Occupation,
    nova_scotia_wage,

    RANK() OVER (
        ORDER BY nova_scotia_wage DESC
    ) AS wage_rank

FROM occupation_scorecard

ORDER BY wage_rank;
"""

result = pd.read_sql(query, conn)

result

,noc_code,Occupation,nova_scotia_wage,wage_rank
0,411,Professional occupations in law,74.52,1
1,311,Health treating and consultation services prof...,52.50,2
2,421,Occupations in front-line public protection se...,48.08,3
3,213,Professional occupations in engineering,48.06,4
4,313,Nursing and allied health professionals,48.00,5
5,312,Therapy and assessment professionals,46.63,6
6,212,Professional occupations in applied sciences (...,46.04,7
7,414,Professional occupations in government services,43.94,8
8,412,Professional occupations in education services,43.27,9
9,211,Professional occupations in natural sciences,40.85,10


### Business Interpretation

This query ranks occupational groups in Nova Scotia according to their 2025 median hourly wage using the SQL `RANK()` window function.

**Key Findings**

- **Professional occupations in law** ranked first with the highest median hourly wage (**$74.52/hour**).
- Health-related occupations occupied several of the top positions, including **Health Treating and Consultation Services Professionals**, **Nursing and Allied Health Professionals**, and **Therapy and Assessment Professionals**.
- Engineering occupations also ranked among the highest-paying occupational groups.
- **Finance**, **Social and Community Services**, and **Paraprofessional Legal and Social Services** ranked near the bottom, reflecting comparatively lower compensation levels within Nova Scotia.

The ranking provides a clear view of wage distribution across occupational groups and helps identify high-paying and lower-paying labour-market segments.

#### Question 10
Which occupational groups represent the highest compensation-market concern?

In [19]:
query = """
WITH historical_gap AS (

    SELECT
        noc_code,
        Occupation,
        ROUND(AVG(market_gap_pct), 2) AS avg_gap_2021_2025

    FROM occupation_trends

    WHERE year BETWEEN 2021 AND 2025

    GROUP BY
        noc_code,
        Occupation
),

growth_analysis AS (

    SELECT
        noc_code,

        ROUND(
            (
                MAX(CASE WHEN year = 2025 THEN nova_scotia_wage END) -
                MAX(CASE WHEN year = 2015 THEN nova_scotia_wage END)
            )
            /
            MAX(CASE WHEN year = 2015 THEN nova_scotia_wage END)
            * 100,
            2
        ) AS ns_growth_pct,

        ROUND(
            (
                MAX(CASE WHEN year = 2025 THEN canada_wage END) -
                MAX(CASE WHEN year = 2015 THEN canada_wage END)
            )
            /
            MAX(CASE WHEN year = 2015 THEN canada_wage END)
            * 100,
            2
        ) AS canada_growth_pct

    FROM occupation_trends

    GROUP BY noc_code
)

SELECT
    s.noc_code,
    s.Occupation,
    s.market_gap_pct AS gap_2025,
    h.avg_gap_2021_2025,
    g.ns_growth_pct,
    g.canada_growth_pct,

    CASE
        WHEN
            s.market_gap_pct < -10
            AND h.avg_gap_2021_2025 < -10
        THEN 'High Priority'

        WHEN
            s.market_gap_pct < -5
            OR h.avg_gap_2021_2025 < -5
        THEN 'Moderate Priority'

        ELSE 'Lower Priority'
    END AS compensation_priority

FROM occupation_scorecard AS s

JOIN historical_gap AS h
    ON s.noc_code = h.noc_code

JOIN growth_analysis AS g
    ON s.noc_code = g.noc_code

ORDER BY
    CASE
        WHEN s.market_gap_pct < -10
             AND h.avg_gap_2021_2025 < -10 THEN 1
        WHEN s.market_gap_pct < -5
             OR h.avg_gap_2021_2025 < -5 THEN 2
        ELSE 3
    END,
    s.market_gap_pct ASC;
"""

result = pd.read_sql(query, conn)

result

,noc_code,Occupation,gap_2025,avg_gap_2021_2025,ns_growth_pct,canada_growth_pct,compensation_priority
0,111,Professional occupations in finance,-15.81,-15.93,35.68,33.58,High Priority
1,413,Professional occupations in social and communi...,-9.85,-9.76,15.39,33.33,Moderate Priority
2,412,Professional occupations in education services,-9.55,-9.46,27.00,29.23,Moderate Priority
3,211,Professional occupations in natural sciences,-8.96,-4.06,22.56,24.99,Moderate Priority
4,212,Professional occupations in applied sciences (...,-8.80,-10.32,36.82,38.64,Moderate Priority
5,213,Professional occupations in engineering,-7.58,-8.79,26.31,26.18,Moderate Priority
6,312,Therapy and assessment professionals,-4.84,-7.78,37.15,32.43,Moderate Priority
7,421,Occupations in front-line public protection se...,-4.64,-5.76,23.28,23.37,Moderate Priority
8,112,Professional occupations in business,-1.86,-9.32,50.97,26.67,Moderate Priority
9,422,"Paraprofessional occupations in legal, social,...",0.30,-7.79,54.29,34.60,Moderate Priority


### Business Interpretation

This query combines current market position, persistent five-year market gaps, and wage-growth context to prioritize occupational groups for compensation review.

**Key Findings**

- **Professional occupations in finance** were classified as **High Priority** because they remain substantially below the Canadian benchmark both currently and over the recent five-year period.
- Several occupations were classified as **Moderate Priority**, indicating either a meaningful current market gap, a persistent historical gap, or both.
- Business and paraprofessional occupations illustrate the importance of considering historical performance alongside current results. Their 2025 market positions are relatively strong, but their five-year average gaps remain notably negative.
- Government services, health-related occupations, and law were classified as **Lower Priority** because their current and historical market positions are comparatively stronger.

This multi-factor approach provides a more robust framework for identifying compensation areas that may warrant further review than relying on a single-year market comparison.

# Dashboard Dataset

## Purpose

This section creates a consolidated dataset for Power BI.

The dashboard dataset combines:

- Current compensation benchmark
- Market position
- Historical market gap
- Wage growth
- Compensation priority

This creates a single reporting table for Power BI.

In [30]:
import pandas as pd

import pandas as pd

# Load the occupation scorecard
# It already contains wage growth measures
scorecard = pd.read_csv(
    "../data/processed/occupation_compensation_scorecard.csv"
)

# Load historical market-gap data
history = pd.read_csv(
    "../data/processed/occupation_market_trends.csv"
)

In [31]:
# Create the historical average (2021–2025)
history_summary = (
    history[
        history["year"].between(2021, 2025)
    ]
    .groupby(
        ["noc_code", "Occupation"],
        as_index=False
    )["market_gap_pct"]
    .mean()
    .rename(
        columns={
            "market_gap_pct": "avg_gap_2021_2025"
        }
    )
)

In [32]:
#Merge the historical summary with the scorecard
dashboard = scorecard.merge(
    history_summary[
        [
            "noc_code",
            "avg_gap_2021_2025"
        ]
    ],
    on="noc_code",
    how="left"
)

In [33]:
#Create Compensation Priority
def priority(row):

    if (
        row["market_gap_pct"] < -10
        and row["avg_gap_2021_2025"] < -10
    ):
        return "High Priority"

    elif (
        row["market_gap_pct"] < -5
        or row["avg_gap_2021_2025"] < -5
    ):
        return "Moderate Priority"

    else:
        return "Lower Priority"


dashboard["compensation_priority"] = dashboard.apply(
    priority,
    axis=1
)

In [34]:
dashboard.columns.tolist()

['noc_code',
 'Occupation',
 'nova_scotia_wage',
 'canada_wage',
 'market_gap_pct',
 'market_position',
 'ns_growth_pct',
 'canada_growth_pct',
 'growth_difference',
 'avg_gap_2021_2025',
 'compensation_priority']

In [35]:
#Reorder columns
dashboard = dashboard[
    [
        "noc_code",
        "Occupation",
        "nova_scotia_wage",
        "canada_wage",
        "market_gap_pct",
        "market_position",
        "avg_gap_2021_2025",
        "ns_growth_pct",
        "canada_growth_pct",
        "growth_difference",
        "compensation_priority"
    ]
]

In [36]:
dashboard.head()

,noc_code,Occupation,nova_scotia_wage,canada_wage,market_gap_pct,market_position,avg_gap_2021_2025,ns_growth_pct,canada_growth_pct,growth_difference,compensation_priority
0,111,Professional occupations in finance,36.20,43.00,-15.81,Below Market,-15.932,35.68,33.58,2.10,High Priority
1,112,Professional occupations in business,40.64,41.41,-1.86,Near Market,-9.324,50.97,26.67,24.30,Moderate Priority
2,211,Professional occupations in natural sciences,40.85,44.87,-8.96,Below Market,-4.060,22.56,24.99,-2.43,Moderate Priority
3,212,Professional occupations in applied sciences (...,46.04,50.48,-8.80,Below Market,-10.320,36.82,38.64,-1.82,Moderate Priority
4,213,Professional occupations in engineering,48.06,52.00,-7.58,Below Market,-8.794,26.31,26.18,0.13,Moderate Priority


In [37]:
dashboard.to_csv(
    "../data/processed/powerbi_compensation_dashboard.csv",
    index=False
)

print("Dashboard dataset created successfully.")

Dashboard dataset created successfully.
